Inferenza con i LLM Instruct: chat templates, few-shot learning e COT
======================================================


Questo notebook analizza l'uso dei modelli di tipo instruct, l'uso dei chat templates e due tipologie di tecniche di promting (few-shot e COT), con il modello frontando tre modelli appartenenti a tre famiglie diverse, [Llama 3.2 1B](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct), [Gemma 3 1B it](https://huggingface.co/google/gemma-3-1b-it) e [Qwen 3 4B](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507).

INSTALLAZIONE E IMPORT
============================================================================

Per eseguire questo notebook, installare prima le dipendenze:

pip install transformers torch bitsanbytes

Su google Colab sono già pre-installate

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.6 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import warnings
warnings.filterwarnings('ignore')

## Usiamo un instruction prompt su un modello base

In [ ]:
from google.colab import userdata

device = 'cuda'
token = userdata.get('HF_TOKEN')

# --- LLAMA 3.2 ---
print("Caricamento Llama 3.2 1B ...")
llama_model_name = "meta-llama/Llama-3.2-1B"

llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name,
    trust_remote_code=True,
    token = token
)

llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    device_map=device,
    trust_remote_code=True,
    token = token
)

Caricamento Llama 3.2 1B ...


In [ ]:
prompt = """Di seguito è riportata un'istruzione che descrive un'attività, abbinata a un input che fornisce ulteriori informazioni contestuali. Scrivi una risposta che completi in modo appropriato la richiesta.

### Istruzione:
Classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.

### Input:
Adoro la pasta.

### Risposta:
"""

print(prompt)

Di seguito è riportata un'istruzione che descrive un'attività, abbinata a un input che fornisce ulteriori informazioni contestuali. Scrivi una risposta che completi in modo appropriato la richiesta.

### Istruzione:
Classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.

### Input:
Adoro la pasta.

### Risposta:



In [ ]:
# definiamo una funzione per fare inferenza ad un LLM

def ask(message, tokenizer, model, device):

    # Tokenizzazione: converte testo in tensori di ID
    inputs = tokenizer(
        message,
        return_tensors="pt"  # Restituisce tensori PyTorch
    )
    inputs['input_ids'] = inputs['input_ids'].to(device)
    inputs['attention_mask'] = inputs['attention_mask'].to(device)

    gen_config = GenerationConfig(
        do_sample=True,
        max_new_tokens=256,
        temperature=0.7,
        pad_token_id=tokenizer.pad_token_id
    )

    response = model.generate(
        **inputs,
        generation_config=gen_config)

    answer_full = tokenizer.decode(response[0], skip_special_tokens=False)
    answer = tokenizer.decode(response[0][len(inputs['input_ids'][0]):], skip_special_tokens=True)

    return answer_full, answer

In [ ]:
answer_full, answer = ask(prompt, llama_tokenizer, llama_model, device)

print(answer_full)

<|begin_of_text|>Di seguito è riportata un'istruzione che descrive un'attività, abbinata a un input che fornisce ulteriori informazioni contestuali. Scrivi una risposta che completi in modo appropriato la richiesta.

### Istruzione:
Classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.

### Input:
Adoro la pasta.

### Risposta:
classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.
```
def classify(text):
    if text >= 0:
        return 'positive'
    elif text <= 0:
        return 'negative'
    else:
        return 'neutral'
```
### Risposta:
classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.
```
def classify(text):
    if text >= 0:
     

In [ ]:
# --- LLAMA 3.2 ---
print("Caricamento Llama 3.2 1B Instruct...")
llama_model_name = "meta-llama/Llama-3.2-1B-Instruct"

llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name,
    trust_remote_code=True,
    token = token
)

llama_tokenizer.pad_token = llama_tokenizer.eos_token

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    device_map=device,
    trust_remote_code=True,
    token = token
)

answer_full, answer = ask(prompt, llama_tokenizer, llama_model, device)

print(answer_full)


Caricamento Llama 3.2 1B Instruct...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

<|begin_of_text|>Di seguito è riportata un'istruzione che descrive un'attività, abbinata a un input che fornisce ulteriori informazioni contestuali. Scrivi una risposta che completi in modo appropriato la richiesta.

### Istruzione:
Classifica il testo inserito in base alla polarità del testo. Rispondi solo con l'etichetta della classe. Le etichette possibili sono: “positivo”, ‘negativo’ e “neutro”.

### Input:
Adoro la pasta.

### Risposta:
“Positivo”.<|eot_id|>


In [ ]:
# Carichiamo entrambi i modelli per confrontare i loro chat templates.
# I modelli Instruct sono addestrati con formati specifici per conversazioni.

print("Caricamento modelli...\n")
device = 'cuda'

# --- LLAMA 3.2 ---
print("Caricamento Llama 3.2 1B Instruct...")
llama_model_name = "meta-llama/Llama-3.2-1B-Instruct"

llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name,
    trust_remote_code=True,
    token = token
)

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_model_name,
    device_map=device,
    trust_remote_code=True,
    token = token
)

# Impostazione pad token se non presente
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token

print(f"✓ Llama caricato")

# --- GEMMA 3 ---
print("Caricamento Gemma 3 1B Instruct...")
gemma_model_name = 'google/gemma-3-1b-it'

gemma_tokenizer = AutoTokenizer.from_pretrained(
    gemma_model_name,
    trust_remote_code=True,
    token = token
)

gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_model_name,
    device_map=device,
    trust_remote_code=True,
    token = token
)

print(f"✓ Gemma caricato")


# --- QWEN 3 ---
print("\nCaricamento Qwen 3 4B Instruct...")
qwen_model_name = "Qwen/Qwen3-4B-Instruct-2507"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_model_name,
    trust_remote_code=True
)

from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True, # carica il modello in 8/4 bit
    bnb_8bit_compute_dtype=torch.float16, # tipo di dati per le operazioni di calcolo (FP16 per velocità)
    bnb_8bit_quant_type="nf4", # tipo di quantizzazione (NF4 è ottimale per distribuzioni di peso normalizzate)
    bnb_8bit_use_double_quant=True, # usa la double quantization per maggiore precisione
)

# Caricamento modello con ottimizzazioni e quantizzazione
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=quantization_config, # Applica la configurazione di quantizzazione
    device_map=device,  # Distribuzione automatica su GPU/CPU
    trust_remote_code=True,
    dtype=torch.float16 # Spesso utile per le restanti operazioni in half-precision
)

print(f"✓ Qwen caricato")

Caricamento modelli...

Caricamento Llama 3.2 1B Instruct...


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✓ Qwen caricato


## Token speciali nei template

I token speciali sono marker che delimitano ruoli e messaggi.

Ogni modello usa token diversi per strutturare la conversazione.

In [ ]:
print(llama_tokenizer.SPECIAL_TOKENS_ATTRIBUTES)
print(gemma_tokenizer.SPECIAL_TOKENS_ATTRIBUTES)
print(qwen_tokenizer.SPECIAL_TOKENS_ATTRIBUTES)

['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']
['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens', 'boi_token', 'eoi_token', 'image_token']
['bos_token', 'eos_token', 'unk_token', 'sep_token', 'pad_token', 'cls_token', 'mask_token', 'additional_special_tokens']


In [ ]:
print("\n🦙 LLAMA - Token speciali:")
for attr_name in llama_tokenizer.SPECIAL_TOKENS_ATTRIBUTES:
    token = getattr(llama_tokenizer, attr_name, "N/A") # Usa getattr per accedere all'attributo
    print(f"  {attr_name}: {token}")


🦙 LLAMA - Token speciali:
  bos_token: <|begin_of_text|>
  eos_token: <|eot_id|>
  unk_token: None
  sep_token: None
  pad_token: <|eot_id|>
  cls_token: None
  mask_token: None
  additional_special_tokens: []


In [ ]:
print("\n✨ Gemma - Token speciali:")
for attr_name in gemma_tokenizer.SPECIAL_TOKENS_ATTRIBUTES:
    token = getattr(gemma_tokenizer, attr_name, "N/A") # Usa getattr per accedere all'attributo
    print(f"  {attr_name}: {token}")


✨ Gemma - Token speciali:
  bos_token: <bos>
  eos_token: <eos>
  unk_token: <unk>
  sep_token: None
  pad_token: <pad>
  cls_token: None
  mask_token: None
  additional_special_tokens: []
  boi_token: <start_of_image>
  eoi_token: <end_of_image>
  image_token: <image_soft_token>


In [ ]:
print("\n🤖 QWEN - Token speciali:")
for attr_name in qwen_tokenizer.SPECIAL_TOKENS_ATTRIBUTES:
    token = getattr(qwen_tokenizer, attr_name, "N/A") # Usa getattr per accedere all'attributo
    print(f"  {attr_name}: {token}")


🤖 QWEN - Token speciali:
  bos_token: None
  eos_token: <|im_end|>
  unk_token: None
  sep_token: None
  pad_token: <|endoftext|>
  cls_token: None
  mask_token: None
  additional_special_tokens: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']


I `Chat template` sono formati standardizzati che i modelli Instruct si aspettano.
Ogni modello ha il proprio formato memorizzato nel tokenizer.

Struttura base:
- `system`
    - Definisce il comportamento e le caratteristiche del modello
   - Imposta il contesto generale
   - Definisce personalità, stile, limitazioni
   - Esempio: "Sei un assistente utile e preciso"

- `user`
    - Rappresenta i messaggi dell'utente, ovvero domande o richieste
   - Input che richiede una risposta

- `assistant`
    - Rappresenta le risposte del modello
   - Usato per esempi di conversazioni precedenti
   - Durante l'inferenza, il modello genera questo ruolo

## Inferenze con un messaggio semplice

In [ ]:
# Creiamo un messaggio semplice
messages_simple = [
    {"role": "user", "content": "Qual è la capitale dell'Italia?"}
]

print("Messaggi input:")
print(messages_simple)

# Applicazione template Llama
print("\n🦙 LLAMA - Template applicato:")
llama_prompt_simple = llama_tokenizer.apply_chat_template(
    messages_simple,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(llama_prompt_simple)
print("\n" + "="*80)

# Applicazione template Gemma
print("\n✨ Gemma - Template applicato:")
gemma_prompt_simple = gemma_tokenizer.apply_chat_template(
    messages_simple,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(gemma_prompt_simple)
print("\n" + "="*80)

# Applicazione template Qwen
print("\n🤖 QWEN - Template applicato:")
qwen_prompt_simple = qwen_tokenizer.apply_chat_template(
    messages_simple,
    tokenize=False,
    add_generation_prompt=True
)
print(qwen_prompt_simple)
print("\n" + "="*80)

Messaggi input:
[{'role': 'user', 'content': "Qual è la capitale dell'Italia?"}]

🦙 LLAMA - Template applicato:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Nov 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

Qual è la capitale dell'Italia?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




✨ Gemma - Template applicato:
<bos><start_of_turn>user
Qual è la capitale dell'Italia?<end_of_turn>
<start_of_turn>model



🤖 QWEN - Template applicato:
<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant




In [ ]:
llama_answer_full, llama_answer = ask(llama_prompt_simple, llama_tokenizer, llama_model, device)

print(llama_answer_full)
print(f'\n\n*** Risposta generata -> {llama_answer}')

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Nov 2025

<|eot_id|><|start_header_id|>user<|end_header_id|>

Qual è la capitale dell'Italia?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

La capitale dell'Italia è Roma.<|eot_id|>


*** Risposta generata -> La capitale dell'Italia è Roma.


In [ ]:
gemma_answer_full, gemma_answer = ask(gemma_prompt_simple, gemma_tokenizer, gemma_model, device)

print(gemma_answer_full)
print(f'\n\n*** Risposta generata -> {gemma_answer}')

`generation_config` default values have been modified to match model-specific defaults: {'top_k': 64, 'top_p': 0.95, 'bos_token_id': 2, 'eos_token_id': [1, 106]}. If this is not desired, please set these values explicitly.


<bos><bos><start_of_turn>user
Qual è la capitale dell'Italia?<end_of_turn>
<start_of_turn>model
La capitale dell'Italia è Roma.<end_of_turn>


*** Risposta generata -> La capitale dell'Italia è Roma.


In [ ]:
qwen_answer_full, qwen_answer = ask(qwen_prompt_simple, qwen_tokenizer, qwen_model, device)

print(qwen_answer_full)
print(f'\n\n*** Risposta generata -> {qwen_answer}')

`generation_config` default values have been modified to match model-specific defaults: {'top_k': 20, 'top_p': 0.8, 'bos_token_id': 151643, 'eos_token_id': [151645, 151643]}. If this is not desired, please set these values explicitly.


<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant
La capitale dell'Italia è **Roma**. ✅<|im_end|>


*** Risposta generata -> La capitale dell'Italia è **Roma**. ✅


## Inferenze con un messaggio contenente il ruolo system

In [ ]:
# Creiamo un messaggio con il ruolo system
messages_system = [
    {"role": "system", "content": "Sei un assistente esperto di geografia che risponde in modo conciso e preciso."},
    {"role": "user", "content": "Qual è la capitale dell'Italia?"}
]

print("Messaggi input:")
print(messages_system)

# Applicazione template Llama
print("\n🦙 LLAMA - Template applicato:")
llama_prompt_system = llama_tokenizer.apply_chat_template(
    messages_system,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(llama_prompt_system)

llama_answer_full, llama_answer = ask(llama_prompt_system, llama_tokenizer, llama_model, device)

print(f'\n\n*** Risposta generata -> {llama_answer}')

Messaggi input:
[{'role': 'system', 'content': 'Sei un assistente esperto di geografia che risponde in modo conciso e preciso.'}, {'role': 'user', 'content': "Qual è la capitale dell'Italia?"}]

🦙 LLAMA - Template applicato:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Nov 2025

Sei un assistente esperto di geografia che risponde in modo conciso e preciso.<|eot_id|><|start_header_id|>user<|end_header_id|>

Qual è la capitale dell'Italia?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




*** Risposta generata -> La capitale dell'Italia è Roma.


In [ ]:
# Applicazione template Gemma
print("\n✨ Gemma - Template applicato:")
gemma_prompt_system = gemma_tokenizer.apply_chat_template(
    messages_system,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(gemma_prompt_system)

gemma_answer_full, gemma_answer = ask(gemma_prompt_system, gemma_tokenizer, gemma_model, device)

print(f'\n\n*** Risposta generata -> {gemma_answer}')


✨ Gemma - Template applicato:
<bos><start_of_turn>user
Sei un assistente esperto di geografia che risponde in modo conciso e preciso.

Qual è la capitale dell'Italia?<end_of_turn>
<start_of_turn>model



*** Risposta generata -> Roma.


In [ ]:
# Applicazione template Qwen
print("\n🤖 QWEN - Template applicato:")
qwen_prompt_system = qwen_tokenizer.apply_chat_template(
    messages_system,
    tokenize=False,
    add_generation_prompt=True
)
print(qwen_prompt_system)

qwen_answer_full, qwen_answer = ask(qwen_prompt_system, qwen_tokenizer, qwen_model, device)

print(f'\n\n*** Risposta generata -> {qwen_answer}')


🤖 QWEN - Template applicato:
<|im_start|>system
Sei un assistente esperto di geografia che risponde in modo conciso e preciso.<|im_end|>
<|im_start|>user
Qual è la capitale dell'Italia?<|im_end|>
<|im_start|>assistant



*** Risposta generata -> La capitale dell'Italia è Roma.


## Inferenze con un messaggio multi-turn

In [ ]:
# Creiamo un messaggio multi-turn
messages_conversation = [
    {"role": "system", "content": "Sei un tutor di matematica paziente e chiaro."},
    {"role": "user", "content": "Cos'è il teorema di Pitagora?"},
    {"role": "assistant", "content": "Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²"},
    {"role": "user", "content": "Puoi farmi un esempio pratico?"}
]

print("Messaggi input:")
print(messages_conversation)

# Applicazione template Llama
print("\n🦙 LLAMA - Template applicato:")
llama_prompt_conversation = llama_tokenizer.apply_chat_template(
    messages_conversation,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(llama_prompt_conversation)

llama_answer_full, llama_answer = ask(llama_prompt_conversation, llama_tokenizer, llama_model, device)

print(f'\n\n*** Risposta generata -> {llama_answer}')

Messaggi input:
[{'role': 'system', 'content': 'Sei un tutor di matematica paziente e chiaro.'}, {'role': 'user', 'content': "Cos'è il teorema di Pitagora?"}, {'role': 'assistant', 'content': "Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²"}, {'role': 'user', 'content': 'Puoi farmi un esempio pratico?'}]

🦙 LLAMA - Template applicato:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Nov 2025

Sei un tutor di matematica paziente e chiaro.<|eot_id|><|start_header_id|>user<|end_header_id|>

Cos'è il teorema di Pitagora?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²<|eot_id|><|start_header_id|>user<|end_header_id|>

Puoi farmi un esempio pratico?<|eot_id|><|start_header_i

In [ ]:
# Applicazione template Gemma
print("\n✨ Gemma - Template applicato:")
gemma_prompt_conversation = gemma_tokenizer.apply_chat_template(
    messages_conversation,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(gemma_prompt_conversation)

gemma_answer_full, gemma_answer = ask(gemma_prompt_conversation, gemma_tokenizer, gemma_model, device)

print(f'\n\n*** Risposta generata -> {gemma_answer}')


✨ Gemma - Template applicato:
<bos><start_of_turn>user
Sei un tutor di matematica paziente e chiaro.

Cos'è il teorema di Pitagora?<end_of_turn>
<start_of_turn>model
Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²<end_of_turn>
<start_of_turn>user
Puoi farmi un esempio pratico?<end_of_turn>
<start_of_turn>model



*** Risposta generata -> Certamente! Il teorema di Pitagora è un concetto fondamentale in geometria e ha molte applicazioni pratiche. Ecco un esempio che lo illustra:

**Esempio:**

Immagina un triangolo rettangolo con cateti di lunghezza 3 e 4.  Possiamo trovare l'ipotenusa.

1. **Definiamo le variabili:**
   * Sia *a* = 3
   * Sia *b* = 4

2. **Applichiamo il teorema di Pitagora:**
   *  a² + b² = c²
   *  3² + 4² = c²
   *  9 + 16 = c²
   *  25 = c²

3. **Risolviamo per c:**
   *  Per trovare il valore di *c*, prendiamo la radice quadrata di entrambi i lati dell'equazione:
 

In [ ]:
# Applicazione template Qwen
print("\n🤖 QWEN - Template applicato:")
qwen_prompt_conversation = qwen_tokenizer.apply_chat_template(
    messages_conversation,
    tokenize=False,
    add_generation_prompt=True
)
print(qwen_prompt_conversation)

qwen_answer_full, qwen_answer = ask(qwen_prompt_conversation, qwen_tokenizer, qwen_model, device)

print(f'\n\n*** Risposta generata -> {qwen_answer}')


🤖 QWEN - Template applicato:
<|im_start|>system
Sei un tutor di matematica paziente e chiaro.<|im_end|>
<|im_start|>user
Cos'è il teorema di Pitagora?<|im_end|>
<|im_start|>assistant
Il teorema di Pitagora afferma che in un triangolo rettangolo, il quadrato dell'ipotenusa è uguale alla somma dei quadrati dei cateti: a² + b² = c²<|im_end|>
<|im_start|>user
Puoi farmi un esempio pratico?<|im_end|>
<|im_start|>assistant



*** Risposta generata -> Certo! Ecco un esempio pratico del teorema di Pitagora.

---

**Problema:**  
Un triangolo rettangolo ha i cateti di lunghezza 3 cm e 4 cm. Quanto misura l’ipotenusa?

---

**Soluzione:**

Sappiamo che nel triangolo rettangolo:

> **a² + b² = c²**

Dove:  
- \( a = 3 \) cm (primo cateto)  
- \( b = 4 \) cm (secondo cateto)  
- \( c \) = ipotenusa (il lato più lungo, opposto all'angolo retto)

Sostituiamo i valori:

\[
3^2 + 4^2 = c^2
\]

\[
9 + 16 = c^2
\]

\[
25 = c^2
\]

Ora facciamo la radice quadrata di entrambi i membri:

\[
c = \sqrt{25} 

## Few-shot prompting

Tecnica di prompting in cui insieme all'istruzione viene fornito uno o più esempi.

Utile al modello per apprendere il task e la struttura dell'output.

In [ ]:
# Creiamo un messaggio per il few-shot prompting
few_shot_messages = [
    {"role": "system", "content": "Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione."},
    {"role": "user", "content": "Potresti dirmi a che ora parte il prossimo treno per Roma?"},
    {"role": "assistant", "content": "Could you tell me what time the next train to Rome leaves"},
    {"role": "user", "content": "Stiamo ascoltando un vecchio disco."},
    {"role": "assistant", "content": "We are listening to an old record."},
    {"role": "user", "content": "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}
    # risposta attesa -> Yesterday I studied for Friday exam, I hope that I will pass it.
]

print("Messaggi input:")
print(few_shot_messages)

# Applicazione template Llama
print("\n🦙 LLAMA - Template applicato:")
llama_prompt_few_shot = llama_tokenizer.apply_chat_template(
    few_shot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(llama_prompt_few_shot)

llama_answer_full, llama_answer = ask(llama_prompt_few_shot, llama_tokenizer, llama_model, device)

print(f'\n\n*** Risposta generata -> {llama_answer}')

Messaggi input:
[{'role': 'system', 'content': "Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione."}, {'role': 'user', 'content': 'Potresti dirmi a che ora parte il prossimo treno per Roma?'}, {'role': 'assistant', 'content': 'Could you tell me what time the next train to Rome leaves'}, {'role': 'user', 'content': 'Stiamo ascoltando un vecchio disco.'}, {'role': 'assistant', 'content': 'We are listening to an old record.'}, {'role': 'user', 'content': "Ieri ho studiato per l'esame di venerdì, spero di passarlo."}]

🦙 LLAMA - Template applicato:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 09 Nov 2025

Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione.<|eot_id|><|start_header_id|>user<|end_header_id|>

Potresti dirmi a che ora parte il prossimo treno per Roma?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Could you tell me what time the next train to Rome lea

In [ ]:
# Applicazione template Gemma
print("\n✨ Gemma - Template applicato:")
gemma_prompt_few_shot = gemma_tokenizer.apply_chat_template(
    few_shot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(gemma_prompt_few_shot)

gemma_answer_full, gemma_answer = ask(gemma_prompt_few_shot, gemma_tokenizer, gemma_model, device)

print(f'\n\n*** Risposta generata -> {gemma_answer}')


✨ Gemma - Template applicato:
<bos><start_of_turn>user
Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione.

Potresti dirmi a che ora parte il prossimo treno per Roma?<end_of_turn>
<start_of_turn>model
Could you tell me what time the next train to Rome leaves<end_of_turn>
<start_of_turn>user
Stiamo ascoltando un vecchio disco.<end_of_turn>
<start_of_turn>model
We are listening to an old record.<end_of_turn>
<start_of_turn>user
Ieri ho studiato per l'esame di venerdì, spero di passarlo.<end_of_turn>
<start_of_turn>model



*** Risposta generata -> Yesterday I studied for the exam on Friday, I hope I pass it.


In [ ]:
# Applicazione template Qwen
print("\n🤖 QWEN - Template applicato:")
qwen_prompt_few_shot = qwen_tokenizer.apply_chat_template(
    few_shot_messages,
    tokenize=False,
    add_generation_prompt=True
)
print(qwen_prompt_few_shot)

qwen_answer_full, qwen_answer = ask(qwen_prompt_few_shot, qwen_tokenizer, qwen_model, device)

print(f'\n\n*** Risposta generata -> {qwen_answer}')


🤖 QWEN - Template applicato:
<|im_start|>system
Sei un traduttore dall'italiano all'inglese. Rispondi solo con la traduzione.<|im_end|>
<|im_start|>user
Potresti dirmi a che ora parte il prossimo treno per Roma?<|im_end|>
<|im_start|>assistant
Could you tell me what time the next train to Rome leaves<|im_end|>
<|im_start|>user
Stiamo ascoltando un vecchio disco.<|im_end|>
<|im_start|>assistant
We are listening to an old record.<|im_end|>
<|im_start|>user
Ieri ho studiato per l'esame di venerdì, spero di passarlo.<|im_end|>
<|im_start|>assistant



*** Risposta generata -> I studied for the Friday exam yesterday, I hope I pass it.


## Chain of Thoughts

Tecnica di prompting in cui viene chiesto al modello di *ragionare* e restituire la sua predizione e i passaggi logici necessari per giungere alla predizione.

Utile al modello per i task più complessi.

In [ ]:
# Creiamo un messaggio per il cot, chiederemo al modello di classificare un testo dal data set BBC-text
# Multiclass dataset of BBC news divided in 5 categories with reference to their topic.
# [Dataset reference](https://www.kaggle.com/datasets/yufengdev/bbc-fulltext-and-category/code)

istruzione = """You are a text classifier. Follow these steps:\n
1. Identify key topics and entities in the article\n
2. Consider which category best matches these topics\n
3. Provide the category: business, politics, tech, sport, or entertainment\n

Format your response as:\n
Reasoning: [your analysis]\n
Category: [category name]\n"""

text = "ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festive comedy christmas with the kranks.  ocean s twelve box office triumph marks the fourth-biggest opening for a december release in the us  after the three films in the lord of the rings trilogy. the sequel narrowly beat its 2001 predecessor  ocean s eleven which took $38.1m (£19.8m) on its opening weekend and $184m (£95.8m) in total. a remake of the 1960s film  starring frank sinatra and the rat pack  ocean s eleven was directed by oscar-winning director steven soderbergh. soderbergh returns to direct the hit sequel which reunites clooney  pitt and roberts with matt damon  andy garcia and elliott gould. catherine zeta-jones joins the all-star cast.  it s just a fun  good holiday movie   said dan fellman  president of distribution at warner bros. however  us critics were less complimentary about the $110m (£57.2m) project  with the los angeles times labelling it a  dispiriting vanity project . a milder review in the new york times dubbed the sequel  unabashedly trivial ."
# entertainment è l'etichetta attesa

cot_messages = [
    {"role": "system", "content": istruzione},
    {"role": "user", "content": text},
]

print("Messaggi input:")
print(cot_messages)

# Applicazione template Llama
print("\n🦙 LLAMA - Template applicato:")
llama_prompt_cot = llama_tokenizer.apply_chat_template(
    cot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(llama_prompt_cot)

llama_answer_full, llama_answer = ask(llama_prompt_cot, llama_tokenizer, llama_model, device)

print(f'\n\n*** Risposta generata -> {llama_answer}')

Messaggi input:
[{'role': 'system', 'content': 'You are a text classifier. Follow these steps:\n\n1. Identify key topics and entities in the article\n\n2. Consider which category best matches these topics\n\n3. Provide the category: business, politics, tech, sport, or entertainment\n\n\nFormat your response as:\n\nReasoning: [your analysis]\n\nCategory: [category name]\n'}, {'role': 'user', 'content': 'ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and fest

In [ ]:
# Applicazione template Gemma
print("\n✨ Gemma - Template applicato:")
gemma_prompt_cot = gemma_tokenizer.apply_chat_template(
    cot_messages,
    tokenize=False,  # Non tokenizzare, mostra solo il testo
    add_generation_prompt=True  # Aggiunge il prompt per la generazione
)
print(gemma_prompt_cot)

gemma_answer_full, gemma_answer = ask(gemma_prompt_cot, gemma_tokenizer, gemma_model, device)

print(f'\n\n*** Risposta generata -> {gemma_answer}')


✨ Gemma - Template applicato:
<bos><start_of_turn>user
You are a text classifier. Follow these steps:

1. Identify key topics and entities in the article

2. Consider which category best matches these topics

3. Provide the category: business, politics, tech, sport, or entertainment


Format your response as:

Reasoning: [your analysis]

Category: [category name]


ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festive comedy christmas with the kranks.

In [ ]:
# Applicazione template Qwen
print("\n🤖 QWEN - Template applicato:")
qwen_prompt_cot = qwen_tokenizer.apply_chat_template(
    cot_messages,
    tokenize=False,
    add_generation_prompt=True
)
print(qwen_prompt_cot)

qwen_answer_full, qwen_answer = ask(qwen_prompt_cot, qwen_tokenizer, qwen_model, device)

print(f'\n\n*** Risposta generata -> {qwen_answer}')


🤖 QWEN - Template applicato:
<|im_start|>system
You are a text classifier. Follow these steps:

1. Identify key topics and entities in the article

2. Consider which category best matches these topics

3. Provide the category: business, politics, tech, sport, or entertainment


Format your response as:

Reasoning: [your analysis]

Category: [category name]
<|im_end|>
<|im_start|>user
ocean s twelve raids box office ocean s twelve  the crime caper sequel starring george clooney  brad pitt and julia roberts  has gone straight to number one in the us box office chart.  it took $40.8m (£21m) in weekend ticket sales  according to studio estimates. the sequel follows the master criminals as they try to pull off three major heists across europe. it knocked last week s number one  national treasure  into third place. wesley snipes  blade: trinity was in second  taking $16.1m (£8.4m). rounding out the top five was animated fable the polar express  starring tom hanks  and festive comedy christm